# Notebook 2: Fine-tune Qwen2.5-3B as Guide Model (LoRA)

**Goal:** Fine-tune Qwen2.5-3B-Instruct on our 2,817 SG pairs so it permanently learns to generate clean step-by-step plans.

**Method:** LoRA in float16 (no 4-bit needed — P100 has enough VRAM)

**Input:** `sg_valid.jsonl` from Notebook 1

**Output:** LoRA adapter weights saved to `/kaggle/working/guide_model/`

---
### ⚠️ Before Running:
1. Add Notebook 1's output as input data:
   - Right panel → **Add Data** → **Notebook Output Files** → select your Notebook 1
   - This makes `sg_valid.jsonl` accessible
2. Enable **GPU P100** and **Internet**
3. Add `HF_TOKEN` in Kaggle Secrets if not already done

In [1]:
# # ── CELL 1: Install ───────────────────────────────────────────
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q trl==0.10.1
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# ── CELL 2: HuggingFace Login ─────────────────────────────────
from huggingface_hub import login
login("")
print('✅ HuggingFace login done')

✅ HuggingFace login done


In [3]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 11.9 MB/s eta 0:00:00a 0:00:01


In [4]:
# ── CELL 3: Imports + GPU Check ───────────────────────────────
import os, json, glob, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/guide_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB


In [5]:
# ── CELL 4: Config ────────────────────────────────────────────

CONFIG = {
    # Model
    "model_name"         : "meta-llama/Llama-3.2-3B-Instruct",

    # LoRA
    "lora_r"             : 16,
    "lora_alpha"         : 32,
    "lora_dropout"       : 0.05,

    # Training
    "num_epochs"         : 3,
    "per_device_batch"   : 4,
    "grad_accumulation"  : 4,       # effective batch = 4 × 4 = 16
    "learning_rate"      : 2e-4,
    "max_seq_length"     : 512,
    "warmup_ratio"       : 0.05,
    "eval_split"         : 0.05,    # 5% held out for validation

    # Output
    "output_dir"         : OUTPUT_DIR,
    "save_steps"         : 100,
    "logging_steps"      : 20,
}

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")

Config:
  model_name            : meta-llama/Llama-3.2-3B-Instruct
  lora_r                : 16
  lora_alpha            : 32
  lora_dropout          : 0.05
  num_epochs            : 3
  per_device_batch      : 4
  grad_accumulation     : 4
  learning_rate         : 0.0002
  max_seq_length        : 512
  warmup_ratio          : 0.05
  eval_split            : 0.05
  output_dir            : /kaggle/working/guide_model
  save_steps            : 100
  logging_steps         : 20


In [6]:
# ── CELL 5: Load sg_valid.jsonl ───────────────────────────────
# Tries both: Notebook 1 output added as dataset, OR same session working dir

def find_sg_file():
    # Option 1: added as Kaggle input dataset from Notebook 1
    patterns = [
        "/kaggle/input/datasets/sufiantabdullah/sg-valid-svamp-lama/sg_valid.jsonl",
        "/kaggle/input/*/sg_valid.jsonl",
    ]
    for pattern in patterns:
        matches = glob.glob(pattern)
        if matches:
            return matches[0]

    # Option 2: same session (unlikely in new notebook but just in case)
    local = "/kaggle/working/sg_data/sg_valid.jsonl"
    if os.path.exists(local):
        return local

    return None


sg_path = find_sg_file()

if sg_path is None:
    print("❌ sg_valid.jsonl not found!")
    print("   Go to: Add Data → Notebook Output Files → select Notebook 1")
    raise FileNotFoundError("sg_valid.jsonl not found")

print(f"Found: {sg_path}")

# Load
raw_data = []
with open(sg_path) as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line))

print(f"Loaded {len(raw_data)} samples")
print(f"\nExample:")
print(f"  Q  : {raw_data[0]['question'][:80]}...")
print(f"  SG : {raw_data[0]['sg_clean'][:120]}...")

Found: /kaggle/input/datasets/sufiantabdullah/sg-valid-svamp-lama/sg_valid.jsonl
Loaded 671 samples

Example:
  Q  : 46 campers went rowing on a day. 43 campers went rowing in the morning and some ...
  SG : Step 1: Determine the initial number of campers who went rowing that day.
Step 2: Subtract the morning rowers to find ou...


In [7]:
# ── CELL 6: Format into Training Pairs ───────────────────────
# Format: system prompt + question → clean SG steps
# We use the SAME system prompt as Notebook 1 so the model learns
# to produce this format reliably without heavy prompting later

SYSTEM_PROMPT = """You are a math problem decomposition assistant.
Your job: break a math problem into 2-5 numbered solution steps.

RULES:
1. Use format: Step 1: ... Step 2: ... etc.
2. Each step must be under 15 words.
3. Do NOT perform calculations or write numbers from computation.
4. Do NOT write the final answer.
5. Write ONLY the steps. Stop immediately after the last step.

EXAMPLE:
Problem: John earns $10/hour and works 8 hours. What does he earn?
Step 1: Identify the hourly rate and total hours worked.
Step 2: Multiply the hourly rate by the number of hours.
Step 3: The result is the total earnings."""


def format_sample(item: dict) -> dict:
    """Format one sample as a chat conversation for SFTTrainer."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Problem: {item['question']}"},
        {"role": "assistant", "content": item["sg_clean"]},
    ]
    return {"messages": messages}


formatted = [format_sample(item) for item in raw_data]

# Train / eval split
import random
random.seed(42)
random.shuffle(formatted)

n_eval  = max(50, int(len(formatted) * CONFIG["eval_split"]))
n_train = len(formatted) - n_eval

train_dataset = Dataset.from_list(formatted[:n_train])
eval_dataset  = Dataset.from_list(formatted[n_train:])

print(f"Train samples : {len(train_dataset)}")
print(f"Eval  samples : {len(eval_dataset)}")
print(f"\nFormatted example (assistant turn):")
print(formatted[0]["messages"][2]["content"])

Train samples : 621
Eval  samples : 50

Formatted example (assistant turn):
Step 1: Determine initial tomato quantity before any were picked.
Step 2: Subtract the number of tomatoes picked yesterday from the initial amount.
Step 3: Calculate remaining tomatoes after yesterday's pickings.
Step 4: Consider if more tomatoes will be removed (today).
Step 5: Finalize calculation to find remaining tomatoes after both days' picks.


In [8]:
# ── CELL 7: Load Qwen2.5-3B in float16 ───────────────────────
# No 4-bit — plain float16 works fine on P100 17GB
# LoRA only trains a tiny fraction of parameters

print(f"Loading {CONFIG['model_name']}...")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code = True,
)
tokenizer.padding_side = "right"    # right padding for training (opposite of inference)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    dtype         = torch.float16,
    device_map    = "auto",
    trust_remote_code = True,
)
model.config.use_cache = False          # required for gradient checkpointing
model.enable_input_require_grads()      # required for LoRA with gradient checkpointing

used_gb = torch.cuda.memory_allocated() / 1e9
print(f"Base model loaded: {used_gb:.2f} GB used")
print(f"Headroom for training: {17.1 - used_gb:.1f} GB")

Loading meta-llama/Llama-3.2-3B-Instruct...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Base model loaded: 3.20 GB used
Headroom for training: 13.9 GB


In [9]:
# ── CELL 8: Apply LoRA ────────────────────────────────────────

lora_config = LoraConfig(
    task_type    = TaskType.CAUSAL_LM,
    r            = CONFIG["lora_r"],
    lora_alpha   = CONFIG["lora_alpha"],
    lora_dropout = CONFIG["lora_dropout"],
    # Qwen2.5 attention + MLP layers
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias         = "none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify memory after LoRA
used_gb = torch.cuda.memory_allocated() / 1e9
print(f"\nAfter LoRA: {used_gb:.2f} GB used")
print("✅ Ready for training")

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

After LoRA: 3.25 GB used
✅ Ready for training


In [10]:
# ── CELL 9: Train ─────────────────────────────────────────────
# Expected time: ~1.5-2 hours for 3 epochs on P100
# Watch eval_loss — should drop steadily

sft_config = SFTConfig(
    output_dir                  = CONFIG["output_dir"],
    num_train_epochs            = CONFIG["num_epochs"],
    per_device_train_batch_size = CONFIG["per_device_batch"],
    per_device_eval_batch_size  = CONFIG["per_device_batch"],
    gradient_accumulation_steps = CONFIG["grad_accumulation"],
    learning_rate               = CONFIG["learning_rate"],
    warmup_ratio                = CONFIG["warmup_ratio"],
    lr_scheduler_type           = "cosine",
    logging_steps               = CONFIG["logging_steps"],
    save_steps                  = CONFIG["save_steps"],
    eval_strategy               = "steps",
    eval_steps                  = CONFIG["save_steps"],
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    fp16                        = True,
    gradient_checkpointing      = True,
    report_to                   = "none",
)

trainer = SFTTrainer( 
    model           = model,
    args            = sft_config,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    processing_class         = tokenizer,
)

print("Starting training...")
print(f"  Epochs          : {CONFIG['num_epochs']}")
print(f"  Train samples   : {len(train_dataset)}")
print(f"  Effective batch : {CONFIG['per_device_batch'] * CONFIG['grad_accumulation']}")
print(f"  Steps/epoch     : {len(train_dataset) // (CONFIG['per_device_batch'] * CONFIG['grad_accumulation'])}")
print("-" * 50)

trainer.train()

print("\nTraining complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/621 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting training...
  Epochs          : 3
  Train samples   : 621
  Effective batch : 16
  Steps/epoch     : 38
--------------------------------------------------


Step,Training Loss,Validation Loss
100,0.208048,0.278578



Training complete!


In [11]:
# ── CELL 10: Save LoRA Adapter ────────────────────────────────

adapter_path = f"{CONFIG['output_dir']}/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
 
print(f"Adapter saved to: {adapter_path}")
print("\nFiles saved:")
for f in os.listdir(adapter_path):
    size = os.path.getsize(f"{adapter_path}/{f}") / 1e6
    print(f"  {f:40s} {size:.1f} MB")

Adapter saved to: /kaggle/working/guide_model/final_adapter

Files saved:
  chat_template.jinja                      0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                97.3 MB
  tokenizer_config.json                    0.0 MB
  tokenizer.json                           17.2 MB
  README.md                                0.0 MB


In [12]:
# ── CELL 11: Quick Eval — Base vs Fine-tuned ──────────────────
# Compares 10 held-out questions: does fine-tuned produce better SGs?

import re

def generate_sg(mdl, tok, question, max_new_tokens=150):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Problem: {question}"},
    ]
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs, 
            max_new_tokens     = max_new_tokens,
            temperature        = 0.1,
            do_sample          = True,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.2,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()


def count_valid_steps(text):
    lines = text.strip().split("\n")
    steps = [l for l in lines if re.match(r"^Step\s*\d+[:.]", l.strip(), re.IGNORECASE)]
    return len(steps)


# Use last 10 eval samples
test_samples = [eval_dataset[i] for i in range(min(10, len(eval_dataset)))]
test_questions = [
    s["messages"][1]["content"].replace("Problem: ", "")
    for s in test_samples
]

print("=" * 60)
print("EVAL: BASE vs FINE-TUNED (10 questions)")
print("=" * 60)

ft_valid = 0

for i, question in enumerate(test_questions):
    ft_output = generate_sg(model, tokenizer, question)
    ft_steps  = count_valid_steps(ft_output)
    ft_ok     = 2 <= ft_steps <= 5
    if ft_ok:
        ft_valid += 1

    print(f"\n[{i+1}] Q: {question[:70]}...")
    print(f"     Fine-tuned ({ft_steps} steps, {'✅' if ft_ok else '❌'}):")
    for line in ft_output.strip().split("\n")[:5]:
        if line.strip():
            print(f"       {line.strip()}")

print("\n" + "=" * 60)
print(f"Fine-tuned valid rate : {ft_valid}/10 ({ft_valid*10}%)")
print("=" * 60)
print("\nNotebook 2 complete.")
print(f"Adapter weights at   : {adapter_path}")
print("Next: Notebook 3 — Generate SCORE critique-correction data")

EVAL: BASE vs FINE-TUNED (10 questions)

[1] Q: Jack received 6 emails in the morning and 8 emails in the afternoon Ho...
     Fine-tuned (5 steps, ✅):
       Step 1: Determine the number of emails received each time period (morning, afternoon).
       Step 2: Subtract the number of emails received in the morning from those received in the afternoon.
       Step 3: Calculate the difference between the two quantities.
       Step 4: Find out how much greater it was for the afternoon compared to the morning.
       Step 5: Express this as an inequality if necessary, otherwise provide just the numerical value.

[2] Q: Randy has 97 blocks. He uses some blocks to build a tower. If there ar...
     Fine-tuned (5 steps, ✅):
       Step 1: Determine initial block count before building the tower.
       Step 2: Subtract remaining blocks from original amount for used blocks.
       Step 3: Calculate difference between initial and leftover blocks.
       Step 4: Express calculation as an equation

In [13]:
import shutil
import os

folder_path = "/kaggle/working/guide_model/final_adapter"
zip_path = "/kaggle/working/final_adapter"  # no .zip here

shutil.make_archive(zip_path, "zip", folder_path)

print("Zip created at:", zip_path + ".zip")

Zip created at: /kaggle/working/final_adapter.zip
